[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/PRiSM_trackB_closedloop_simulation_colab.ipynb)

# PRiSM Track B (Modular, Colab Thin Notebook)

This notebook is intentionally thin: core logic lives in `src/trackb/*.py`.


## Run Contract

1. Sync repo in Colab runtime.
2. Run setup cell on a fresh runtime (`RUN_SETUP=True`).
3. Initialize config + persistence settings.
4. Build dataset, run preflight/calibration, pass surprise gate.
5. Run closed-loop simulation, summarize, export artifacts.


In [ ]:
# Sync this repo into the current Colab runtime
import os, subprocess, sys

REPO_URL = 'https://github.com/achyutmorang/thesis-research-colab.git'
REPO_DIR = '/content/thesis-research-colab'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.check_call(['git', 'clone', '--branch', 'main', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo synced at:', REPO_DIR)


In [ ]:
# Runtime setup (set RUN_SETUP=True on a fresh runtime, otherwise keep False)
import os, subprocess, sys
from pathlib import Path

RUN_SETUP = False

if RUN_SETUP:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'jax[cuda12]>=0.7.0,<0.8', '-f', 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'mediapy', 'seaborn', 'scikit-learn', 'tqdm'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'hydra-core==1.3.2', 'omegaconf==2.3.0', 'lightning==2.3.3', 'pytorch-lightning==2.3.3', 'einops==0.8.0', 'transformers==4.46.3', 'huggingface_hub'])

    if not Path('/content/LatentDriver/.git').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Sephirex-X/LatentDriver.git', '/content/LatentDriver'])

if '/content/LatentDriver' not in sys.path:
    sys.path.insert(0, '/content/LatentDriver')
os.environ['PYTHONPATH'] = '/content/LatentDriver:' + os.environ.get('PYTHONPATH', '')

# Optional manual checkpoint fallback
ckpt_path = Path('/content/checkpoints/lantentdriver_t2_J3.ckpt')
if not ckpt_path.exists():
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    print('[ckpt] missing, you can manually download to', ckpt_path)

print('Setup done. RUN_SETUP =', RUN_SETUP)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from src.trackb import (
    initialize_configs,
    configure_persistent_run_prefix,
    restore_artifacts_via_upload,
    make_waymax_data_iter,
    build_trackb_runner_and_splits,
    run_preflight_and_calibration,
    run_surprise_quality_gate,
    run_trackb_closed_loop,
    summarize_method_outputs,
    export_trackb_artifacts,
)

cfg, search_cfg, latentdriver_ckpt_scan_df = initialize_configs()

# Runtime persistence + shard controls
n_shards = 1
shard_id = 0
run_tag = 'prism_trackB_run'
persist_root = '/content/drive/MyDrive/prism_trackB'
restore_from_upload = False

run_prefix = configure_persistent_run_prefix(cfg, run_tag, persist_root, shard_id, n_shards)
print('[run-prefix]', run_prefix)
print('[shard]', f'{shard_id+1}/{max(1,n_shards)}')

if restore_from_upload:
    _ = restore_artifacts_via_upload(
        cfg.run_prefix,
        required_keys=['per_scenario_results', 'per_eval_trace', 'thresholds', 'closedloop_calibration'],
    )

if isinstance(latentdriver_ckpt_scan_df, pd.DataFrame) and len(latentdriver_ckpt_scan_df) > 0:
    display(latentdriver_ckpt_scan_df[['name', 'path', 'size_mb', 'mtime_utc', 'score']].head(20))


In [ ]:
# Build data iterator and dataset splits
dataset_config, data_iter = make_waymax_data_iter(cfg)

(
    runner,
    data,
    train_idx,
    test_idx,
    eval_idx_all,
    eval_idx,
    reference_df,
    base_eval_openloop_df,
) = build_trackb_runner_and_splits(
    cfg=cfg,
    data_iter=data_iter,
    dataset_config=dataset_config,
    n_shards=n_shards,
    shard_id=shard_id,
)

print('Loaded scenarios:', len(data['scenarios']))
print('Train/Test/Eval(all)/Eval(run):', len(train_idx), len(test_idx), len(eval_idx_all), len(eval_idx))
state_count = int(sum('state' in data['scenarios'][int(i)] for i in eval_idx))
print('Eval scenarios with retained simulator state:', state_count, '/', len(eval_idx))

display(base_eval_openloop_df.head())


In [ ]:
# Preflight + calibration (resume-aware)
(
    preflight_df,
    closedloop_calib_df,
    trackb_thresholds,
    calib_diag_df,
    calib_quant_df,
) = run_preflight_and_calibration(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=restore_from_upload,
)

print('Track B thresholds source:', trackb_thresholds.get('source', 'unknown'))
print(trackb_thresholds)

print('Track B preflight checks:')
display(preflight_df)
print('Calibration diagnostics:')
display(calib_diag_df)
display(calib_quant_df)
display(closedloop_calib_df.head())


In [ ]:
# Surprise quality gate (hard stop on degenerate surprise)
gate_summary, dist_change_summary = run_surprise_quality_gate(closedloop_calib_df)
print('Surprise gate diagnostics:')
display(gate_summary)
print('Per-step distribution change stats:')
display(dist_change_summary)
print('Surprise diagnostics gate PASSED.')


In [ ]:
# Main closed-loop simulation (resume-aware)
static_frames = {
    'base_eval_openloop_df': base_eval_openloop_df,
    'reference_df': reference_df,
    'closedloop_calib_df': closedloop_calib_df,
    'preflight_df': preflight_df,
    'calib_diag_df': calib_diag_df,
    'calib_quant_df': calib_quant_df,
}

trackb_results_df, trackb_trace_df = run_trackb_closed_loop(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    thresholds=trackb_thresholds,
    run_prefix=cfg.run_prefix,
    static_frames=static_frames,
)

print('Track B result rows:', len(trackb_results_df))
print('Track B per-eval trace rows:', len(trackb_trace_df))
display(trackb_results_df.head())


In [ ]:
# Summaries
quick_summary_df, sanity_df, fairness_checks_df, trace_diag_df = summarize_method_outputs(
    trackb_results_df, trackb_trace_df
)

print('Quick summary (planner-dependent surprise):')
display(quick_summary_df)
print('Sanity checks:')
display(sanity_df)
print('Fairness checks:')
display(fairness_checks_df)
print('Per-eval trace diagnostics:')
display(trace_diag_df)

plt.figure(figsize=(9, 4))
sns.barplot(data=quick_summary_df, x='method', y='failure_rate')
plt.title('Track B: Failure Rate by Method (Closed-Loop)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Export all artifacts
artifact_paths = export_trackb_artifacts(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    trackb_thresholds=trackb_thresholds,
    quick_summary_df=quick_summary_df,
    sanity_df=sanity_df,
    fairness_checks_df=fairness_checks_df,
    trace_diag_df=trace_diag_df,
)

print('Saved artifacts:')
for k, p in artifact_paths.items():
    print(f' - {k}: {p}')
